This notebook deals with combining different datasets downloaded from  
<https://data.marine.copernicus.eu/product/MEDSEA_MULTIYEAR_PHY_006_004/download>.

We have downloaded **2 × 6 datasets**:  
- 6 datasets with monthly values from **01/01/1987 to 01/05/2023**  
- 6 datasets with interim monthly values from **01/06/2023 to 01/05/2025**  

The variables are **Temperature** and **Salinity**, each at the following depths:  
- 51.38 m  
- 303.56 m  
- 1005.14 m  

To be able to properly use this notebook as intended, organize your folders as follows:
- Place the **older datasets** in a folder named `reanalysis`  
- Place the **newer datasets** in a folder named `forecasting`  


In [3]:
import os
import xarray as xr
import cartopy.crs as ccrs
import matplotlib as plt
import numpy as np

def combine_folder(path):
    folder_path = os.path.abspath(path)
    datasets = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".nc"):
            file_path = os.path.join(folder_path, filename)
            try:
                ds = xr.open_dataset(file_path)
                datasets.append(ds)
            except Exception as e:
                print(f"Error with {filename}: {e}")

    combined = xr.combine_by_coords(datasets, combine_attrs="override")
    return combined


Simply read each dataset seperately.

In [ ]:
ds_forecast = combine_folder("forecasting")

In [6]:
ds_reanalysis = combine_folder("reanalysis")

Here we ensure that both datasets have exactly the same depths and coordinates.  
For example, one dataset might list a depth as **51.38 m**, while the other uses **51.37779 m**.  

In [ ]:
ds_forecast = ds_forecast.sel(
    longitude=ds_reanalysis.longitude,
    latitude=ds_reanalysis.latitude,
    depth=ds_reanalysis.depth,
    method="nearest",
    tolerance=1e-3
)

ds_forecast = ds_forecast.assign_coords(
    longitude=ds_reanalysis.longitude,
    latitude=ds_reanalysis.latitude
)

Here we simply remove the Atlantic Ocean and the Black Sea from our data, since there are not relevant for the later analysis

In [8]:
import xarray as xr
lon2d, lat2d = np.meshgrid(
    ds_forecast.longitude,
    ds_forecast.latitude
)

atlantic_mask = ~((lon2d < 0) & (lat2d > 41))
blacksea_mask = ~((lon2d > 27) & (lat2d > 41))
mask = atlantic_mask & blacksea_mask

mask_da = xr.DataArray(
    mask,
    coords={'latitude': ds_forecast.latitude, 'longitude': ds_forecast.longitude},
    dims=['latitude', 'longitude']
)

ds_forecast = ds_forecast.where(mask_da)


Let's check whether we have the correct timestamps.  

In [24]:
print(ds_reanalysis.time)
print()
print(ds_forecast.time)


<xarray.DataArray 'time' (time: 437)> Size: 3kB
array(['1987-01-01T00:00:00.000000000', '1987-02-01T00:00:00.000000000',
       '1987-03-01T00:00:00.000000000', ..., '2023-03-01T00:00:00.000000000',
       '2023-04-01T00:00:00.000000000', '2023-05-01T00:00:00.000000000'],
      shape=(437,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 3kB 1987-01-01 1987-02-01 ... 2023-05-01
Attributes:
    standard_name:  time
    long_name:      Time
    axis:           T

<xarray.DataArray 'time' (time: 24)> Size: 192B
array(['2023-06-01T00:00:00.000000000', '2023-07-01T00:00:00.000000000',
       '2023-08-01T00:00:00.000000000', '2023-09-01T00:00:00.000000000',
       '2023-10-01T00:00:00.000000000', '2023-11-01T00:00:00.000000000',
       '2023-12-01T00:00:00.000000000', '2024-01-01T00:00:00.000000000',
       '2024-02-01T00:00:00.000000000', '2024-03-01T00:00:00.000000000',
       '2024-04-01T00:00:00.000000000', '2024-05-01T00:00:00.000000000',
       '2024-06-01T00:0

Now we coarsen the data by a factor of 2 for both longitude and latitude  
to reduce the dimensionality and the file size:

In [ ]:
ds_reanalysis_coarse = ds_reanalysis.coarsen(
    longitude=2,
    latitude=2,
    boundary='trim'
).mean()

In [ ]:
ds_forecast_coarse = ds_forecast.coarsen(
    longitude=2,
    latitude=2,
    boundary='trim'
).mean()

Here we concatenate the two datasets along the **time** dimension,  
since we have already ensured that they are equivalent in all other dimensions.

In [19]:
ds_combined = xr.concat([ds_reanalysis_coarse, ds_forecast_coarse], dim="time")

In [20]:
print(ds_combined)

<xarray.Dataset> Size: 1GB
Dimensions:    (time: 461, depth: 3, latitude: 190, longitude: 508)
Coordinates:
  * time       (time) datetime64[ns] 4kB 1987-01-01 1987-02-01 ... 2025-05-01
  * depth      (depth) float32 12B 51.38 303.6 1.005e+03
  * latitude   (latitude) float32 760B 30.21 30.29 30.38 ... 45.79 45.88 45.96
  * longitude  (longitude) float32 2kB -5.979 -5.896 -5.812 ... 36.1 36.19 36.27
Data variables:
    so         (time, depth, latitude, longitude) float32 534MB nan nan ... nan
    thetao     (time, depth, latitude, longitude) float32 534MB nan nan ... nan
Attributes:
    Conventions:       CF-1.11
    title:             Salinity (3D) - Monthly Mean
    institution:       Centro Euro-Mediterraneo sui Cambiamenti Climatici - C...
    source:            MFS E3R1
    contact:           servicedesk.cmems@mercator-ocean.eu
    references:        Please check in CMEMS catalogue the INFO section for p...
    comment:           Please check in CMEMS catalogue the INFO section f

To save the dataset, we need to specify how missing values (land data) should be handled:

In [64]:
for var in ds_combined.data_vars:
    ds_combined[var].encoding['_FillValue'] = 1.0e20  # oder 1.0e20
    ds_combined[var].attrs.pop('missing_value', None)

ds_combined.to_netcdf("medsea1987to2025_train.nc")

Now we want to create another dataset by **artificially generating new data** using linear interpolation.  
We take all 460 existing timestamps and, for every consecutive pair, calculate a linearly interpolated value for every feature at every depth.  
To place these new timestamps in the middle of each interval, we add a **15-day offset**.  
This results in **459 new timestamps**.  


In [58]:
import pandas as pd

new_time = pd.date_range(
    start=ds_combined['time'].values[0] + pd.offsets.Day(15),
    end=ds_combined['time'].values[-1] - pd.offsets.Day(15),
    freq='MS'
) + pd.offsets.Day(14)

print(new_time)

DatetimeIndex(['1987-02-15', '1987-03-15', '1987-04-15', '1987-05-15',
               '1987-06-15', '1987-07-15', '1987-08-15', '1987-09-15',
               '1987-10-15', '1987-11-15',
               ...
               '2024-07-15', '2024-08-15', '2024-09-15', '2024-10-15',
               '2024-11-15', '2024-12-15', '2025-01-15', '2025-02-15',
               '2025-03-15', '2025-04-15'],
              dtype='datetime64[ns]', length=459, freq=None)


In [60]:
ds_interp = ds_combined.interp(time=new_time)

In [63]:
for var in ds_interp.data_vars:
    ds_interp[var].encoding['_FillValue'] = 1.0e20  # oder 1.0e20
    ds_interp[var].attrs.pop('missing_value', None)

ds_combined.to_netcdf("medsea1987to2025_val.nc")